# Dow Momentum Study v0.1

固定済みの研究スナップショットだけを用いる、探索的な後ろ向きイベントスタディです。確定スイング安値割れに既存モメンタムを超える追加情報があるか、警報なし対照群と比較します。古典的ダウ理論の厳密な再現、独立したアウトオブサンプル検証、売買戦略の検証成功とは呼びません。

**新しい市場データを取得せず、元のParquetと既存Alpha Engineを変更しません。** 原本に対する1306.Tの試験的補正は解析用コピーだけに適用します。実際の固定Driveデータ・Colab実機での検証は、合成データのテストとは別です。操作・計測定義は `docs/dow_momentum_study_v01.md` を参照してください。


## 1. 固定データを保存したGoogle Driveを接続

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. 設定（編集はこのセル）

初期値は指定済みの2026年11月4日のスナップショットです。入力が見つからなくてもダウンロードで代替しません。

主分析は事前固定：SPAN=5、安値割れ閾値0%、クールダウン21観測日、将来5/21/63観測日（主評価21）。感度分析は主設定と分けて保存します。`BOOTSTRAP_SAMPLES` と `SEED` は計算量・再現性の設定です。結果を見て主分析を変更しないでください。


In [ ]:
REPO_URL = 'https://github.com/avalon19920629-create/Alpha-Engine-backtest.git'
CODE_REF = 'codex/dow-momentum-study-v0-1'
EXPECTED_BASE_SHA = '57d99dbc75fe69fd59dd031fc4df78ce3dff18a2'
SNAPSHOT_DIR = '/content/drive/MyDrive/alpha_engine/research_runs/20261104T124443Z_residual100_n6_9b9d99b92dcf'
OUTPUT_ROOT = '/content/drive/MyDrive/alpha_engine/analysis/dow_momentum_v01'
BOOTSTRAP_SAMPLES = 500
SEED = 20261104


## 3. 入力の存在確認・研究コードのSHA固定・依存関係導入

新しい `/content` 作業ディレクトリへGitコードを取得します。これは市場データの取得ではありません。既存チェックアウトへのpull・reset、研究スナップショットの更新は行いません。指定起点コミットの子孫であることと、保護対象の既存コードが不変であることを確認してから起動します。


In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

snapshot = Path(SNAPSHOT_DIR).expanduser().resolve()
output = Path(OUTPUT_ROOT).expanduser().resolve()
if not snapshot.is_dir():
    raise FileNotFoundError(f'固定スナップショットが見つかりません: {snapshot}。新規取得では代替しません。')
if output == snapshot or snapshot in output.parents:
    raise ValueError('分析出力先は入力スナップショットの外側にしてください。')
required = (
    'price_data/price_cache.parquet', 'price_data/benchmark_prices.parquet',
    'price_data/price_cache_metadata.json', 'price_data/price_data_quality.csv',
    'research_run_metadata.json', 'engine_outputs/score_components.csv',
)
for name in required:
    if not (snapshot / name).is_file():
        raise FileNotFoundError(f'固定入力が不足しています: {snapshot / name}')

REPO_DIR = Path(tempfile.mkdtemp(prefix='dow-momentum-study-', dir='/content'))
subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)

def resolve_code_ref(ref):
    for candidate in (ref, f'refs/remotes/origin/{ref}'):
        result = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'rev-parse', '--verify', '--end-of-options',
             f'{candidate}^{{commit}}'], text=True, capture_output=True,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'clone済み参照を解決できません: {ref}')

CODE_SHA = resolve_code_ref(CODE_REF)
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'merge-base', '--is-ancestor', EXPECTED_BASE_SHA, CODE_SHA],
    check=True,
)
protected = (
    'Alpha-Engine.py', 'alpha_engine_backtest.py', 'alpha_engine_live_screener.py',
    'alpha_engine_production.py', 'alpha_engine_order_planner.py', 'alpha_engine_ttl_manager.py',
    'core_alpha_integration_backtest.py', 'alpha_engine_research_runner.py',
    'Alpha_Engine_Research_Operations.ipynb', 'requirements.txt', 'requirements-research.txt',
)
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'diff', '--exit-code', EXPECTED_BASE_SHA, CODE_SHA, '--', *protected],
    check=True,
)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', CODE_SHA], check=True)
print('Analysis code ref / pinned SHA:', CODE_REF, CODE_SHA)
print('Existing research base SHA:', EXPECTED_BASE_SHA)
print('Fixed snapshot:', snapshot)
print('Analysis output root:', output)
assert (REPO_DIR / 'dow_momentum_study.py').is_file()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-dow-study.txt')],
    check=True,
)


## 4. 固定スナップショットを解析

既存の `load_research_prices()` で価格ハッシュを照合してから解析します。新しい研究実行やLive Screenerは起動しません。全候補ユニバースを使い、最終選定6銘柄や11月4日のランキングを過去イベントの対象選定に使用しません。

別Pythonプロセスで実行し、結果通知ファイルも毎回新しく作ります。失敗時に前回成功したフォルダを再表示しません。実データ量により計算に時間がかかることがあります。


In [ ]:
import json

STUDY_RECEIPT = Path(tempfile.mkdtemp(
    prefix='dow-study-receipt-', dir=str(REPO_DIR.parent),
)) / 'result.json'
run_config = {
    'snapshot_dir': str(snapshot), 'output_root': str(output),
    'bootstrap_samples': BOOTSTRAP_SAMPLES, 'seed': SEED,
}
child_code = r"""
import json
import sys
from pathlib import Path
from dow_momentum_study import run_study

receipt = Path(sys.argv[2])
try:
    run_dir = run_study(**json.loads(sys.argv[1]))
except Exception as exc:
    receipt.write_text(json.dumps({
        'call_status': 'failed', 'run_dir': str(getattr(exc, 'run_dir', '') or ''),
        'error': str(exc),
    }, ensure_ascii=False), encoding='utf-8')
    raise
else:
    receipt.write_text(json.dumps({'call_status': 'returned', 'run_dir': str(run_dir)}), encoding='utf-8')
"""
result = subprocess.run(
    [sys.executable, '-c', child_code, json.dumps(run_config), str(STUDY_RECEIPT)],
    cwd=REPO_DIR,
)
receipt = json.loads(STUDY_RECEIPT.read_text()) if STUDY_RECEIPT.exists() else {
    'call_status': 'failed', 'run_dir': '',
    'error': '今回の結果通知がありません。上のエラーを確認してください。',
}
RUN_DIR = Path(receipt['run_dir']) if receipt.get('run_dir') else None
print('Process exit code:', result.returncode)
print('Study receipt:', receipt)
if result.returncode:
    print('解析は失敗しました。部分成果物は完成済みの研究結果として扱わないでください。')


## 5. 検査状態・数値結果・補正履歴を確認

主分析の21観測日後を中心に、件数・マッチング不能・将来未観測・市場別差・信頼区間を確認します。反例と感度分析も確認し、有利な設定だけを抜き出さないでください。銘柄・日付単位の依存性と、試験的な価格補正という制約があります。

`right_censored_count`（将来価格不足）と `excluded_influence_count`（既知異常の影響除外）は別の件数です。影響除外は過去253共通価格観測値＝252リターンの入力とピボット判定窓を確認します。

`event_concentration.csv` の市場下落フラグ・日付別警報数も確認してください。メタデータの `event_concentration` はその要約です。


In [ ]:
import pandas as pd
from IPython.display import JSON, Markdown, display

if RUN_DIR is None:
    raise RuntimeError('今回の分析フォルダが不明です。前回の結果ではなく実行セルのエラーを確認してください。')
meta_path = RUN_DIR / 'study_metadata.json'
if meta_path.is_file():
    study_meta = json.loads(meta_path.read_text(encoding='utf-8'))
    display(JSON(study_meta))
    if study_meta.get('status') != 'completed' or study_meta.get('validation_status') != 'passed':
        print('未完了または検査失敗です。数値を検証済み結果として扱わないでください。')
else:
    print('未作成:', meta_path)

report = RUN_DIR / 'study_report.md'
if report.is_file():
    display(Markdown(report.read_text(encoding='utf-8')))

for name in (
    'summary_statistics.csv', 'uncertainty.csv', 'sensitivity_analysis.csv',
    'correction_log.csv', 'momentum_reproduction.csv', 'data_quality_warnings.csv',
    'control_matches.csv', 'event_outcomes.csv', 'event_concentration.csv',
):
    path = RUN_DIR / name
    display(Markdown(f'**{name}**'))
    if path.is_file():
        try:
            table = pd.read_csv(path)
            print('rows:', len(table))
            display(table.head(20))
        except pd.errors.EmptyDataError:
            print('記録行なし（ファイルは空です）。')
    else:
        print('未作成:', path)


## 6. チャートと保存先

SNDK・DELL・4062.Tの波形は事後選定された予備事例です。全対象の件数推移、警報あり／なしの分布、モメンタム別成績と合わせて読みます。典型例・反例はレポートの決定論的な選定基準を確認してください。

チャートは説明用で、数値・不確実性評価の代わりではありません。本研究の結果が有望でも、既存スコア・売買・運用設定へ自動反映しません。

説明例は観測済み21日後の株価警報から相対リターンの最低・最高を選びます（`lowest_relative_return.svg` / `highest_relative_return.svg`）。最低でも下落、最高でも上昇とは限りません。`rising_after_alert_counterexample.svg` は実際に株式リターンがプラスの例がある場合だけ作り、その中で最大のプラス例を表示します。

`alert_momentum_outcomes.svg` はマッチした観測済み21日後のペアを警報時3か月モメンタム帯で比較します。`momentum_only.svg` は固定観測位置のモメンタム単独比較で、別の抽出方法・分母です。


In [ ]:
from IPython.display import SVG

if RUN_DIR is None:
    raise RuntimeError('今回の分析フォルダがありません。')
chart_paths = sorted((RUN_DIR / 'charts').glob('*.svg'))
print('SVG charts:', len(chart_paths))
for path in chart_paths:
    display(Markdown(f'**{path.name}**'))
    display(SVG(filename=str(path)))
print('Fixed source snapshot (unchanged):', snapshot)
print('This analysis run:', RUN_DIR)
print('Pinned analysis code SHA:', CODE_SHA)
print('Driveで分析フォルダを開き、study_metadata.json・study_report.md・CSV・chartsを確認してください。')
